# Подготовка данных о рынке видеоигр за 2000–2013 годы

**Автор:** Ефимов Алексей

### Цель и задачи

Цель проекта — проверить качество исторических данных о видеоиграх и подготовить срез за 2000–2013 годы для исследования платформ, жанров, региональных продаж и RPG.

Для этого необходимо:

1. Загрузить данные и изучить их структуру.
2. Привести названия столбцов к `snake_case` и проверить типы данных.
3. Обработать пропуски и проверить явные и неявные дубликаты.
4. Отобрать игры, выпущенные с 2000 по 2013 год включительно.
5. Категоризовать оценки пользователей и критиков.
6. Выделить семь платформ с наибольшим числом релизов за выбранный период.

### Описание данных

Файл `new_games.csv` содержит записи о видеоиграх на разных платформах. На платформе Практикума он находится по пути `/datasets/new_games.csv`.

| Исходный столбец | Содержание |
| --- | --- |
| `Name` | Название игры |
| `Platform` | Игровая платформа |
| `Year of Release` | Год выпуска |
| `Genre` | Жанр |
| `NA sales`, `EU sales`, `JP sales`, `Other sales` | Продажи по регионам, млн копий |
| `Critic Score` | Оценка критиков от 0 до 100 |
| `User Score` | Оценка пользователей от 0 до 10 |
| `Rating` | Возрастной рейтинг ESRB |

Одна игра может встречаться на нескольких платформах: такие записи обозначают разные релизы и не считаются дубликатами только из-за совпадения названия.

### Этапы работы

1. Загрузка и первичный обзор данных.
2. Проверка названий столбцов, типов, пропусков и дубликатов.
3. Отбор релизов 2000–2013 годов.
4. Категоризация оценок и выделение топ-7 платформ.
5. Итоговые выводы и ограничения подготовленного набора.

---

## 1. Загрузка данных и знакомство с ними



In [1]:
from pathlib import Path

import pandas as pd

In [2]:
# Путь на платформе Практикума или рядом с тетрадью при локальном запуске.
data_path = next(
    (path for path in (Path('/datasets/new_games.csv'), Path('new_games.csv'))
     if path.exists()),
    None
)
if data_path is None:
    raise FileNotFoundError('Поместите new_games.csv рядом с тетрадью.')

df = pd.read_csv(data_path)

In [3]:
df.head()

,Name,Platform,Year of Release,Genre,NA sales,EU sales,JP sales,Other sales,Critic Score,User Score,Rating
0,Wii Sports,Wii,2006.0,Sports,41.36,28.96,3.77,8.45,76.0,8,E
1,Super Mario Bros.,NES,1985.0,Platform,29.08,3.58,6.81,0.77,NaN,NaN,NaN
2,Mario Kart Wii,Wii,2008.0,Racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,Wii Sports Resort,Wii,2009.0,Sports,15.61,10.93,3.28,2.95,80.0,8,E
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,11.27,8.89,10.22,1.00,NaN,NaN,NaN


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  object 
 1   Platform         16956 non-null  object 
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  object 
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  object 
 6   JP sales         16956 non-null  object 
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  object 
 10  Rating           10085 non-null  object 
dtypes: float64(4), object(7)
memory usage: 1.4+ MB


In [5]:
df.shape

(16956, 11)

In [6]:
df.columns

Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='object')

In [7]:
df.isna().sum()

Name                  2
Platform              0
Year of Release     275
Genre                 2
NA sales              0
EU sales              0
JP sales              0
Other sales           0
Critic Score       8714
User Score         6804
Rating             6871
dtype: int64

Исходная таблица содержит **16 956 строк и 11 столбцов**. Названия полей соответствуют описанию, но записаны в разном стиле. Есть пропуски в годе выпуска, оценках, рейтинге ESRB, названии и жанре. `year_of_release` пока имеет вещественный тип из-за отсутствующих значений. Числовые по смыслу `EU sales`, `JP sales` и `User Score` имеют тип `object`: проверим их содержимое перед преобразованием.

---

## 2. Проверка ошибок и предобработка

### 2.1. Названия столбцов

In [8]:
df.columns

Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='object')

In [9]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r'\s+', '_', regex=True)
)

In [10]:
df.columns

Index(['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales',
       'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating'],
      dtype='object')

Названия всех столбцов приведены к `snake_case`: буквы стали строчными, а пробелы заменены подчёркиваниями.

### 2.2. Типы данных


In [11]:
df.dtypes

name                object
platform            object
year_of_release    float64
genre               object
na_sales           float64
eu_sales            object
jp_sales            object
other_sales        float64
critic_score       float64
user_score          object
rating              object
dtype: object

In [12]:
df['user_score'].unique()

array(['8', nan, '8.3', '8.5', '6.6', '8.4', '8.6', '7.7', '6.3', '7.4',
       '8.2', '9', '7.9', '8.1', '8.7', '7.1', '3.4', '5.3', '4.8', '3.2',
       '8.9', '6.4', '7.8', '7.5', '2.6', '7.2', '9.2', '7', '7.3', '4.3',
       '7.6', '5.7', '5', '9.1', '6.5', 'tbd', '8.8', '6.9', '9.4', '6.8',
       '6.1', '6.7', '5.4', '4', '4.9', '4.5', '9.3', '6.2', '4.2', '6',
       '3.7', '4.1', '5.8', '5.6', '5.5', '4.4', '4.6', '5.9', '3.9',
       '3.1', '2.9', '5.2', '3.3', '4.7', '5.1', '3.5', '2.5', '1.9', '3',
       '2.7', '2.2', '2', '9.5', '2.1', '3.6', '2.8', '1.8', '3.8', '0',
       '1.6', '9.6', '2.4', '1.7', '1.1', '0.3', '1.5', '0.7', '1.2',
       '2.3', '0.5', '1.3', '0.2', '0.6', '1.4', '0.9', '1', '9.7'],
      dtype=object)

In [13]:
for column in ['eu_sales', 'jp_sales', 'user_score']:
    invalid = df[column].notna() & pd.to_numeric(df[column], errors='coerce').isna()
    print(f'{column}: {df.loc[invalid, column].value_counts().to_dict()}')

eu_sales: {'unknown': 6}
jp_sales: {'unknown': 4}
user_score: {'tbd': 2464}


In [14]:
numeric_columns = [
    'year_of_release', 'na_sales', 'eu_sales', 'jp_sales',
    'other_sales', 'critic_score', 'user_score'
]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')

In [15]:
df.dtypes

name                object
platform            object
year_of_release    float64
genre               object
na_sales           float64
eu_sales           float64
jp_sales           float64
other_sales        float64
critic_score       float64
user_score         float64
rating              object
dtype: object

In [16]:
score_limits = {'user_score': (0, 10), 'critic_score': (0, 100)}
for column, (low, high) in score_limits.items():
    invalid = df[column].notna() & ~df[column].between(low, high)
    print(f'{column}: значений вне диапазона — {invalid.sum()}')

for column in ['na_sales', 'eu_sales', 'jp_sales', 'other_sales']:
    print(f'{column}: отрицательных продаж — {(df[column] < 0).sum()}')

user_score: значений вне диапазона — 0
critic_score: значений вне диапазона — 0
na_sales: отрицательных продаж — 0
eu_sales: отрицательных продаж — 0
jp_sales: отрицательных продаж — 0
other_sales: отрицательных продаж — 0


В `eu_sales` и `jp_sales` обнаружено соответственно **6 и 4** значения `unknown`, а в `user_score` — **2 464** значения `tbd` («оценка ещё не определена»). Они стали пропусками при преобразовании через `pd.to_numeric(errors='coerce')`. В итоговой таблице числовые значения продаж и оценок имеют числовые типы. Недопустимых оценок и отрицательных продаж не обнаружено.

После удаления строк без года его можно преобразовать из `float64` в `int64`.

### 2.3. Пропущенные значения

In [17]:
missing_values = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_percent': (df.isna().mean() * 100).round(2)
})

missing_values.sort_values(by='missing_count', ascending=False)

,missing_count,missing_percent
user_score,9268,54.66
critic_score,8714,51.39
rating,6871,40.52
year_of_release,275,1.62
eu_sales,6,0.04
jp_sales,4,0.02
name,2,0.01
genre,2,0.01
platform,0,0.00
na_sales,0,0.00


In [18]:
df[df['name'].isna()]

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
661,NaN,GEN,1993.0,NaN,1.78,0.53,0.00,0.08,NaN,NaN,NaN
14439,NaN,GEN,1993.0,NaN,0.00,0.00,0.03,0.00,NaN,NaN,NaN


In [19]:
initial_rows = len(df)
df = df.dropna(subset=['name', 'genre', 'year_of_release']).copy()
removed_for_missing = initial_rows - len(df)
df['year_of_release'] = df['year_of_release'].astype('int64')

# Значение UNKNOWN отсутствует среди исходных возрастных рейтингов.
assert 'UNKNOWN' not in df['rating'].dropna().astype(str).str.upper().unique()
df['rating'] = df['rating'].fillna('UNKNOWN')

sales_columns = ['na_sales', 'eu_sales', 'jp_sales', 'other_sales']
for column in sales_columns:
    group_mean = df.groupby(['platform', 'year_of_release'])[column].transform('mean')
    df[column] = df[column].fillna(group_mean)

df.isna().sum()

name                  0
platform              0
year_of_release       0
genre                 0
na_sales              0
eu_sales              0
jp_sales              0
other_sales           0
critic_score       8594
user_score         9121
rating                0
dtype: int64

Больше всего пропусков обнаружено в оценках пользователей (9 268, 54,66%), оценках критиков (8 714, 51,39%) и рейтинге ESRB (6 871, 40,52%). Источник не содержит объяснения этих пропусков: оценки могли не быть выставлены, а рейтинг ESRB может отсутствовать, например, у игр для некоторых регионов.

Пропуски обработаны следующим образом:

- Удалены **277** строк без названия, жанра или года. Без года невозможно определить принадлежность к исследуемому периоду; две строки без названия также не имеют жанра.
- После удаления пропусков год приведён к `int64`.
- Пустой `rating` заменён на индикатор `UNKNOWN`, отсутствующий среди исходных значений ESRB. Он означает только отсутствие данных, а не возрастную категорию.
- Десять пропусков в региональных продажах заполнены средним по платформе и году. Для всех десяти значений подходящее среднее нашлось; продажи не заменялись нулём.
- Пропуски в оценках оставлены как `NaN`, чтобы не приписывать играм несуществующие оценки.

### 2.4. Явные и неявные дубликаты

Сравним уникальные обозначения платформ, жанров, рейтингов и годов. Затем нормализуем текст и проверим полные совпадения строк.

In [20]:
df['platform'].sort_values().unique()

array(['2600', '3DO', '3DS', 'DC', 'DS', 'GB', 'GBA', 'GC', 'GEN', 'GG',
       'N64', 'NES', 'NG', 'PC', 'PCFX', 'PS', 'PS2', 'PS3', 'PS4', 'PSP',
       'PSV', 'SAT', 'SCD', 'SNES', 'TG16', 'WS', 'Wii', 'WiiU', 'X360',
       'XB', 'XOne'], dtype=object)

In [21]:
df['genre'].sort_values().unique()

array(['ACTION', 'ADVENTURE', 'Action', 'Adventure', 'FIGHTING',
       'Fighting', 'MISC', 'Misc', 'PLATFORM', 'PUZZLE', 'Platform',
       'Puzzle', 'RACING', 'ROLE-PLAYING', 'Racing', 'Role-Playing',
       'SHOOTER', 'SIMULATION', 'SPORTS', 'STRATEGY', 'Shooter',
       'Simulation', 'Sports', 'Strategy'], dtype=object)

In [22]:
df['rating'].sort_values().unique()

array(['AO', 'E', 'E10+', 'EC', 'K-A', 'M', 'RP', 'T', 'UNKNOWN'],
      dtype=object)

In [23]:
df['year_of_release'].sort_values().unique()

array([1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990,
       1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001,
       2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012,
       2013, 2014, 2015, 2016])

In [24]:
df['name'] = df['name'].str.strip().str.lower()
df['genre'] = df['genre'].str.strip().str.lower()
df['platform'] = df['platform'].str.strip().str.upper()
df['rating'] = df['rating'].str.strip().str.upper()

In [25]:
df['platform'].sort_values().unique()

array(['2600', '3DO', '3DS', 'DC', 'DS', 'GB', 'GBA', 'GC', 'GEN', 'GG',
       'N64', 'NES', 'NG', 'PC', 'PCFX', 'PS', 'PS2', 'PS3', 'PS4', 'PSP',
       'PSV', 'SAT', 'SCD', 'SNES', 'TG16', 'WII', 'WIIU', 'WS', 'X360',
       'XB', 'XONE'], dtype=object)

In [26]:
df['genre'].sort_values().unique()

array(['action', 'adventure', 'fighting', 'misc', 'platform', 'puzzle',
       'racing', 'role-playing', 'shooter', 'simulation', 'sports',
       'strategy'], dtype=object)

In [27]:
df['rating'].sort_values().unique()

array(['AO', 'E', 'E10+', 'EC', 'K-A', 'M', 'RP', 'T', 'UNKNOWN'],
      dtype=object)

In [28]:
duplicates_count = df.duplicated().sum()
duplicates_count

np.int64(235)

In [29]:
df = df.drop_duplicates().reset_index(drop=True)

deleted_rows = initial_rows - len(df)
deleted_percent = deleted_rows / initial_rows * 100

print(f'Удалено строк без названия, жанра или года: {removed_for_missing}')
print(f'Удалено полных дубликатов: {duplicates_count}')
print(f'Всего удалено: {deleted_rows} ({deleted_percent:.2f}%)')
print(f'Осталось строк: {len(df)}')

Удалено строк без названия, жанра или года: 277
Удалено полных дубликатов: 235
Всего удалено: 512 (3.02%)
Осталось строк: 16444


Среди обозначений платформ, жанров и годов нет содержательных вариантов написания, требующих объединения. Для жанров встречается разный регистр; его следует унифицировать перед проверкой повторов. Названия игр и жанры приведены к нижнему регистру, платформы и рейтинги ESRB — к верхнему. Пробелы по краям удалены.

После нормализации найдено и удалено **235 полных дубликатов** (`duplicated()` и `drop_duplicates()`). Игра с одинаковым названием на другой платформе сохраняется как отдельная запись. Всего вместе со строками без обязательных сведений удалено **512 записей (3,02%)**; для дальнейшей работы осталось **16 444 записи**.

После предобработки названия полей и текстовые категории унифицированы, числовые значения приведены к нужным типам, обязательные поля заполнены, а полные дубликаты удалены. Пропущенные оценки сохранены как неизвестные. Для продаж в двух регионах восстановлены десять отдельных значений по платформе и году выпуска; эти значения являются оценками, что важно учитывать в последующем анализе региональных продаж.

---

## 3. Фильтрация по годам

Для исследования оставим релизы с 2000 по 2013 год включительно. Срез сохраним в отдельном датафрейме `df_actual`.

In [30]:
df_actual = df[(df['year_of_release'] >= 2000) & (df['year_of_release'] <= 2013)].copy()

In [31]:
df_actual.shape

(12781, 11)

In [32]:
df_actual['year_of_release'].min(), df_actual['year_of_release'].max()

(np.int64(2000), np.int64(2013))

In [33]:
df_actual.head()

,name,platform,year_of_release,genre,na_sales,eu_sales,jp_sales,other_sales,critic_score,user_score,rating
0,wii sports,WII,2006,sports,41.36,28.96,3.77,8.45,76.0,8.0,E
2,mario kart wii,WII,2008,racing,15.68,12.76,3.79,3.29,82.0,8.3,E
3,wii sports resort,WII,2009,sports,15.61,10.93,3.28,2.95,80.0,8.0,E
6,new super mario bros.,DS,2006,platform,11.28,9.14,6.50,2.88,89.0,8.5,E
7,wii play,WII,2006,misc,13.96,9.18,2.93,2.84,58.0,6.6,E


Датафрейм `df_actual` содержит **12 781 запись** за **2000–2013 годы** включительно. Одна игра, выпущенная на нескольких платформах, может занимать несколько строк.

---

## 4. Категоризация оценок и топ-7 платформ

In [34]:
df_actual['user_score_category'] = pd.cut(
    df_actual['user_score'],
    bins=[0, 3, 8, 10],
    labels=['низкая оценка', 'средняя оценка', 'высокая оценка'],
    right=False,
    include_lowest=True
)

In [35]:
df_actual.loc[df_actual['user_score'] == 10, 'user_score_category'] = 'высокая оценка'

In [36]:
df_actual['user_score_category'] = df_actual['user_score_category'].cat.add_categories('нет оценки')
df_actual['user_score_category'] = df_actual['user_score_category'].fillna('нет оценки')

In [37]:
df_actual['user_score_category'].value_counts()

user_score_category
нет оценки        6298
средняя оценка    4081
высокая оценка    2286
низкая оценка      116
Name: count, dtype: int64

In [38]:
df_actual['critic_score_category'] = pd.cut(
    df_actual['critic_score'],
    bins=[0, 30, 80, 100],
    labels=['низкая оценка', 'средняя оценка', 'высокая оценка'],
    right=False,
    include_lowest=True
)

In [39]:
df_actual.loc[df_actual['critic_score'] == 100, 'critic_score_category'] = 'высокая оценка'

In [40]:
df_actual['critic_score_category'] = df_actual['critic_score_category'].cat.add_categories('нет оценки')
df_actual['critic_score_category'] = df_actual['critic_score_category'].fillna('нет оценки')

In [41]:
df_actual['critic_score_category'].value_counts()

critic_score_category
нет оценки        5612
средняя оценка    5422
высокая оценка    1692
низкая оценка       55
Name: count, dtype: int64

In [42]:
df_actual[['name', 'user_score', 'user_score_category', 'critic_score', 'critic_score_category']].head(10)

,name,user_score,user_score_category,critic_score,critic_score_category
0,wii sports,8.0,высокая оценка,76.0,средняя оценка
2,mario kart wii,8.3,высокая оценка,82.0,высокая оценка
3,wii sports resort,8.0,высокая оценка,80.0,высокая оценка
6,new super mario bros.,8.5,высокая оценка,89.0,высокая оценка
7,wii play,6.6,средняя оценка,58.0,средняя оценка
8,new super mario bros. wii,8.4,высокая оценка,87.0,высокая оценка
10,nintendogs,NaN,нет оценки,NaN,нет оценки
11,mario kart ds,8.6,высокая оценка,91.0,высокая оценка
13,wii fit,7.7,средняя оценка,80.0,высокая оценка
14,kinect adventures!,6.3,средняя оценка,61.0,средняя оценка


Каждая имеющаяся оценка отнесена к одной из трёх категорий: низкая `[0, 3)` / `[0, 30)`, средняя `[3, 8)` / `[30, 80)`, высокая `[8, 10]` / `[80, 100]` для пользователей и критиков соответственно. В коде правые границы высокой категории (10 и 100) обработаны отдельно. Для пропусков сохранён индикатор `нет оценки`; это обозначение отсутствующих данных, а не четвёртый диапазон оценок.

В выбранном периоде нет пользовательской оценки у **6 298 записей**, а оценки критиков — у **5 612**. При дальнейших сравнениях игр по качеству важно учитывать этот большой объём недостающих оценок.

In [43]:
top_7_platforms = (
    df_actual['platform']
    .value_counts()
    .head(7)
)

top_7_platforms

platform
PS2     2127
DS      2120
WII     1275
PSP     1180
X360    1121
PS3     1087
GBA      811
Name: count, dtype: int64

In [44]:
top_7_platforms_list = top_7_platforms.index

In [45]:
df_actual['is_top_7_platform'] = df_actual['platform'].isin(top_7_platforms_list)

In [46]:
df_actual[['name', 'platform', 'is_top_7_platform']].head(10)

,name,platform,is_top_7_platform
0,wii sports,WII,True
2,mario kart wii,WII,True
3,wii sports resort,WII,True
6,new super mario bros.,DS,True
7,wii play,WII,True
8,new super mario bros. wii,WII,True
10,nintendogs,DS,True
11,mario kart ds,DS,True
13,wii fit,WII,True
14,kinect adventures!,X360,True


---

## 5. Итоговый вывод

Исходный набор содержал **16 956 записей и 11 столбцов**. После нормализации названий полей, проверки типов и значений, обработки пропусков и удаления повторов осталось **16 444 записи**. Удалено 277 строк без обязательных сведений и 235 полных дубликатов — всего **512 строк (3,02%)**. Десять недостающих значений региональных продаж восстановлены средним по платформе и году; отсутствующие оценки не заменялись вымышленными значениями.

В отдельный датафрейм `df_actual` отобраны **12 781 релиз** за **2000–2013 годы**. Он содержит исходные поля с именами в `snake_case` и три новых столбца:

| Поле | Содержание |
| --- | --- |
| `user_score_category` | Низкая, средняя или высокая оценка пользователя; `нет оценки` для пропусков |
| `critic_score_category` | Низкая, средняя или высокая оценка критиков; `нет оценки` для пропусков |
| `is_top_7_platform` | `True`, если платформа входит в топ-7 по числу релизов за период |

По количеству записей лидируют **PS2 (2 127)**, **DS (2 120)**, **WII (1 275)**, **PSP (1 180)**, **X360 (1 121)**, **PS3 (1 087)** и **GBA (811)**. Эти числа обозначают релизы на платформах, а не продажи и не число уникальных игр без учёта платформы.

Набор подготовлен для дальнейшего изучения жанров, региональных продаж и RPG (`genre == 'role-playing'`). Из-за большого числа пропусков в оценках выводы о качестве игр по оценкам следует делать с оговоркой о неполноте данных.